# 红利低波 ETF 股息率-国债利差定投

spread = TTM股息率 − 10Y国债利率(百分点)。>3% 加仓, >1.5% 加仓, 1~1.5% 不动, <1% 清仓; 首次 >1.5% 建仓 1 万; 分红当日现金再投。

策略实现见 `strategy/dividend_bond_spread.py`; 本 notebook 只做参数组合、结果检视与画图。

## 可调参数 (默认关闭, 见策略 docstring)
- `cap_mode='legacy'`(默认): 加仓额 = 当前持仓市值的 40%/>3%、20%/1.5~3%(复利滚仓)。
- `cap_mode='signal'`: 加仓到 NAV 目标上限 — >3% 到 `cap_mult` 满仓, 1.5~3% 到 80%。
- `trend_ma=True`: 收盘跌破 200 日均线时 `cap_mult` 降到 1/3。
- `vol_target=0.15`: 按目标波动率倒数缩放 `cap_mult`(风险平价式降仓)。

`cap_mult` 同时约束加仓目标与超限削减, 所以趋势/波动率开关开启时仓位会被主动压回目标。

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_ROOT = Path.cwd().resolve()
while not (_ROOT / 'pyproject.toml').exists():
    _ROOT = _ROOT.parent
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

from backtest.engine import buy_and_hold_nav, dividends_by_day
from backtest.metrics import formatted_perf_row
from dataload.etf import fetch_data, fetch_etf_dividends
from strategy.dividend_bond_spread import INITIAL_CAPITAL, DividendBondSpreadStrategy

In [ ]:
CODE, SYMBOL = '510880', 'sh510880'
START, END = pd.Timestamp('2010-01-01'), pd.Timestamp('2026-10-07')

px = fetch_data(CODE, START, END, adjust='')  # 不复权价, 现金分红另计
divs = fetch_etf_dividends(SYMBOL)
print(f'价格 {px.index[0].date()}~{px.index[-1].date()}  {len(px)} 行;  分红 {len(divs)} 次')

In [ ]:
runs = {
    # A 原版口径: 按当前持仓市值的 40%(>3%)/20%(1.5~3%) 加仓
    '原版(持仓市值 40/20%)': DividendBondSpreadStrategy(
        start=START, end=END, price_df=px, divs=divs).run(),
    # B 出风险口径: NAV 目标上限 + 200 日线趋势 + 波动率目标
    '目标仓位+趋势+波动率': DividendBondSpreadStrategy(
        start=START, end=END, price_df=px, divs=divs,
        cap_mode='signal', trend_ma=True, vol_target=0.15).run(),
}
bench = buy_and_hold_nav(px['close'], dividends_by_day(px.index, divs)).rename('510880 B&H(含分红)')
curve = {name: df['nav'] / INITIAL_CAPITAL for name, df in runs.items()}
curve[bench.name] = bench / INITIAL_CAPITAL

In [ ]:
ax = pd.DataFrame(curve).plot(figsize=(13, 6), grid=True)
ax.set_title(f'{CODE} 股息率-国债利差定投 vs 买入持有');  ax.set_ylabel('净值');  plt.show()

In [ ]:
pd.DataFrame([formatted_perf_row(s.dropna(), name) for name, s in curve.items()])

In [ ]:
monthly = pd.DataFrame({name: s.dropna().resample('ME').last().pct_change().dropna()
                        for name, s in curve.items()})
monthly.to_csv('reports/generated/dividend_bond_spread_monthly.csv')
monthly